# 09 — Linha de base acústica e falsos alarmes (24 h de operação normal)

Usa a sessão de linha de base, já aprovada no notebook 08, para duas tarefas:

1. **Linha de base acústica do nó.** Calcula a média e o desvio de cada um dos 40 filtros mel com o **mesmo espectrograma do firmware** (16 kHz, quadros de 25 ms, passo de 10 ms, 98 × 40). Gera o `base.csv` no formato que o firmware de inferência carrega do microSD, no lugar da calibração de 5 min feita pelo comando `CAL`.
2. **Falsos alarmes em operação normal.** Executa a CNN int8 e o LSTM sobre o trecho de avaliação e conta quantas decisões seriam anomalia ou classe acima de Normal, sabendo que todo o período é normal.

**Separação para não avaliar com os dados da própria calibração:** a linha de base usada na avaliação vem só da **primeira parte** da sessão (`FRACAO_BASE`, padrão 50%); os falsos alarmes são contados na **segunda parte**. O `base.csv` final, gravado no cartão, usa a sessão inteira.

Também compara a linha de base de **5 min** (o que o `CAL` faz) com a da primeira metade, para medir o ganho de uma calibração longa.

**Como usar**

1. Rode antes o notebook 08 nesta sessão.
2. Confira se os modelos estão em `dissertacao_tinyml/modelos/cnn_v3/` e `dissertacao_tinyml/modelos/lstm/` (gerados nos notebooks 03 v3 e 04).
3. Preencha a célula 2 e execute tudo. No Colab, as 24 h levam cerca de 15 a 30 min (leitura de cerca de 2,8 GB e cerca de 43 mil execuções da CNN).

**Saídas** em `dissertacao_tinyml/coleta/linha_de_base/`: `base.csv` (para o microSD), `linha_base_<sessão>.json` e as figuras.

In [14]:
# 1. Ambiente e caminhos
import os, sys, json, subprocess, importlib.util
if importlib.util.find_spec("tensorflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"], check=True)
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
CONJUNTO = os.environ.get("CONJUNTO_TINYML", "coleta_sim")   # "coleta_sim" = sessões sintéticas (notebook 07b) | "coleta" = dados do nó
CONJUNTO = "coleta_sim"
SIM = CONJUNTO != "coleta"
print("Conjunto de dados:", CONJUNTO)
DISS = f"{BASE}/dissertacao_cap9" + ("_sim" if SIM else ""); os.makedirs(DISS, exist_ok=True)   # figuras e tabelas que vão para o texto
COLETA, CNN, LSTM = f"{BASE}/{CONJUNTO}", f"{BASE}/modelos/cnn_v3", f"{BASE}/modelos/lstm"
SAIDA = f"{COLETA}/linha_de_base"; os.makedirs(SAIDA, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 10, "axes.titlesize": 10.5, "savefig.dpi": 300, "savefig.bbox": "tight"})
virgula = plt.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "\u2212"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Conjunto de dados: coleta_sim


In [15]:
# 2. PREENCHA
SESSAO       = "SIM001"     # sessão de linha de base aprovada no notebook 08
FRACAO_BASE  = 0.5        # fração inicial usada como linha de base na avaliação; o restante mede falsos alarmes
JANELAS_CAL  = 300        # 5 min, como o comando CAL do firmware
PASSO_AVAL   = 1          # 1 = todas as janelas de 1 s da avaliação; 2 = uma sim, uma não (mais rápido)
LIMIAR       = 0.5        # firmware: média das 10 últimas p(anômalo) >= 0,5 -> nível acústico 1
N_MEDIA      = 10
PASTA = f"{COLETA}/{SESSAO}"
assert os.path.isdir(PASTA), f"Pasta não encontrada: {PASTA}"
rel = f"{COLETA}/relatorios/relatorio_{SESSAO}.json"
if os.path.exists(rel):
    r08 = json.load(open(rel)); print("Notebook 08:", "APROVADA" if r08.get("aprovada") else "COM PENDÊNCIAS", r08.get("alertas", []))
else:
    print("Atenção: rode antes o notebook 08 nesta sessão.")

Notebook 08: APROVADA []


In [16]:
# 3. Leitura da sessão (mesmas funções do notebook 08)
# ===== Leitura de uma sessão do firmware de coleta (Coleta_InSitu 1.0 / 1.1 / 1.2-diag) =====
import glob, os
import numpy as np
import pandas as pd
import soundfile as sf

FS_COLETA = 16000

def ler_info(pasta):
    """INFO.TXT: linhas chave=valor (início e, se a sessão foi encerrada, totais)."""
    info = {}
    caminho = os.path.join(pasta, "INFO.TXT")
    if os.path.exists(caminho):
        for linha in open(caminho, encoding="latin-1"):
            if "=" in linha:
                k, v = linha.strip().split("=", 1)
                info[k] = v
    return info

def ler_evt(pasta):
    return pd.read_csv(os.path.join(pasta, "EVT.CSV"))

def ler_amb(pasta):
    """AMB.CSV: t_ms, amostra, temp, umid, press, perdidas (campos vazios = BME280 sem leitura)."""
    a = pd.read_csv(os.path.join(pasta, "AMB.CSV"))
    a["h"] = a["amostra"] / FS_COLETA / 3600.0            # horas desde o início do áudio
    return a

def listar_wav(pasta):
    """Arquivos WAV em ordem, com o número de amostras pelo tamanho do arquivo e pelo cabeçalho.
    Cabeçalho com 0 amostras = arquivo não fechado (sessão desligada sem encerramento)."""
    linhas = []
    for arq in sorted(glob.glob(os.path.join(pasta, "A*.WAV")) + glob.glob(os.path.join(pasta, "A*.wav"))):
        tam = os.path.getsize(arq)
        with open(arq, "rb") as f:
            cab = f.read(44)
        dados_cab = int.from_bytes(cab[40:44], "little") if len(cab) == 44 else -1
        linhas.append({"arquivo": os.path.basename(arq), "caminho": arq, "bytes": tam,
                       "amostras_arquivo": max(0, (tam - 44) // 2), "amostras_cabecalho": max(0, dados_cab // 2),
                       "cabecalho_ok": dados_cab == tam - 44})
    w = pd.DataFrame(linhas)
    if len(w):
        w["inicio_amostra"] = w["amostras_arquivo"].cumsum() - w["amostras_arquivo"]
    return w

def ler_pcm(caminho, ini=0, n=None):
    """Lê amostras int16 do WAV pelo tamanho real do arquivo (funciona mesmo com cabeçalho não corrigido)."""
    with open(caminho, "rb") as f:
        f.seek(44 + 2 * ini)
        dados = f.read(-1 if n is None else 2 * n)
    return np.frombuffer(dados[: len(dados) // 2 * 2], dtype="<i2")

def serie_10s(amb):
    """Média das leituras em intervalos de 10 s do áudio (5 leituras de 2 s), como no firmware de inferência."""
    a = amb.dropna(subset=["temp", "umid", "press"]).copy()
    a["bloco"] = (a["amostra"] // (10 * FS_COLETA)).astype(int)
    s = a.groupby("bloco")[["temp", "umid", "press"]].mean()
    s = s.reindex(range(s.index.min(), s.index.max() + 1))   # blocos sem leitura ficam NaN
    s["h"] = s.index * 10 / 3600.0
    return s


In [17]:
# 4. Espectrograma log-mel idêntico ao do firmware (core_mimii do notebook 01 e espectro_mel.h)
FS, JANELA, FRAME, HOP, NFFT, N_MEL, FMIN, FMAX, EPS = 16000, 16000, 400, 160, 512, 40, 20.0, 8000.0, 1e-6
N_FRAMES = 1 + (JANELA - FRAME) // HOP                       # 98
HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FRAME) / FRAME)).astype(np.float32)
def _mel(f): return 2595.0 * np.log10(1.0 + f / 700.0)
def _hz(m): return 700.0 * (10 ** (m / 2595.0) - 1.0)
def banco_mel():
    freqs = np.arange(NFFT // 2 + 1) * FS / NFFT
    p = _hz(np.linspace(_mel(FMIN), _mel(FMAX), N_MEL + 2))
    B = np.zeros((N_MEL, NFFT // 2 + 1), np.float32)
    for m in range(N_MEL):
        B[m] = np.maximum(0, np.minimum((freqs - p[m]) / (p[m + 1] - p[m]), (p[m + 2] - freqs) / (p[m + 2] - p[m + 1])))
    return B
BANCO = banco_mel()
IDX = np.arange(FRAME)[None, :] + HOP * np.arange(N_FRAMES)[:, None]

def log_mel_lote(x):
    """x: (n, 16000) float32 em [-1, 1] -> (n, 98, 40), igual a core_mimii.log_mel janela a janela."""
    q = x[:, IDX] * HANN                                       # (n, 98, 400)
    e = np.fft.rfft(q, n=NFFT, axis=2)
    pot = (e.real ** 2 + e.imag ** 2).astype(np.float32)
    return np.log(pot @ BANCO.T + EPS).astype(np.float32)

def janelas_da_sessao(wav, ini_j=0, fim_j=None, passo=1, lote=600):
    """Gera (índices globais das janelas, log-mel (n, 98, 40)) para as janelas de 1 s em [ini_j, fim_j),
    lendo cada WAV uma única vez. Uma janela pode cruzar a fronteira entre dois arquivos."""
    total = int(wav.amostras_arquivo.sum()) // JANELA
    fim_j = total if fim_j is None else min(fim_j, total)
    resto, j = np.zeros(0, np.int16), 0
    for _, w in wav.iterrows():
        fim_arq = int(w.inicio_amostra + w.amostras_arquivo)
        if fim_arq <= ini_j * JANELA and len(resto) == 0 and fim_arq % JANELA == 0:
            j = fim_arq // JANELA; continue                    # arquivo inteiro antes do trecho pedido
        x = np.concatenate([resto, ler_pcm(w.caminho)])
        n = len(x) // JANELA
        js = np.arange(j, j + n)
        loc = np.where((js >= ini_j) & (js < fim_j) & ((js - ini_j) % passo == 0))[0]
        for k in range(0, len(loc), lote):
            l = loc[k:k + lote]
            bl = np.stack([x[i * JANELA:(i + 1) * JANELA] for i in l]).astype(np.float32) / 32768.0
            yield js[l], log_mel_lote(bl)
        resto, j = x[n * JANELA:], j + n
        if j >= fim_j: break

In [18]:
# 5. Estatísticas por filtro mel (média e desvio populacional + 1e-6, como no firmware e no notebook 03 v2)
info, wav = ler_info(PASTA), listar_wav(PASTA)
assert int(info.get("ganho_pdm", 20)) == 20, "Ganho diferente do firmware de inferência (20)"
N_JAN = int(wav.amostras_arquivo.sum()) // JANELA
J_BASE = int(N_JAN * FRACAO_BASE)
print(f"Janelas de 1 s: {N_JAN} ({N_JAN / 3600:.2f} h); linha de base de avaliação: {J_BASE}; avaliação: {N_JAN - J_BASE}")

class Acum:
    def __init__(s): s.n = 0; s.s = np.zeros(N_MEL); s.s2 = np.zeros(N_MEL)
    def add(s, lm): f = lm.reshape(-1, N_MEL).astype(np.float64); s.n += len(f); s.s += f.sum(0); s.s2 += (f ** 2).sum(0)
    def stats(s):
        mu = s.s / s.n; sd = np.sqrt(np.maximum(s.s2 / s.n - mu ** 2, 0)) + 1e-6
        return mu.astype(np.float32), sd.astype(np.float32)
tot, primeira, cal5 = Acum(), Acum(), Acum()
for js, lm in janelas_da_sessao(wav):
    tot.add(lm)
    s = js < J_BASE
    if s.any(): primeira.add(lm[s])
    s = js < JANELAS_CAL
    if s.any(): cal5.add(lm[s])
BASES = {"sessao_inteira": tot.stats(), "primeira_parte": primeira.stats(), "cal_5min": cal5.stats()}
for k, (mu, sd) in BASES.items(): print(f"{k:15s} média dos filtros {mu.mean():7.3f}  desvio médio {sd.mean():.3f}")

Janelas de 1 s: 86400 (24.00 h); linha de base de avaliação: 43200; avaliação: 43200


KeyboardInterrupt: 

In [ ]:
# 6. base.csv para o microSD (formato lido por carregarBase() no firmware de inferência)
mu, sd = BASES["sessao_inteira"]
with open(f"{SAIDA}/base.csv", "w", newline="\n") as f:
    f.write("filtro,media,desvio\n")
    for m in range(N_MEL): f.write(f"{m},{mu[m]:.6f},{sd[m]:.6f}\n")
print(open(f"{SAIDA}/base.csv").read()[:300], "...")
print("Copie base.csv para a RAIZ do microSD do firmware de inferência.")

In [ ]:
# 7. CNN int8 no trecho de avaliação, com a linha de base da primeira parte e com a de 5 min
qjs = json.load(open(f"{CNN}/cnn_int8_quantizacao.json"))
it = Interpreter(model_content=open(f"{CNN}/cnn_int8.tflite", "rb").read()); it.allocate_tensors()
ent, sai = it.get_input_details()[0], it.get_output_details()[0]
(se, ze), (ss, zs) = ent["quantization"], sai["quantization"]
def prob_cnn(lm, base):
    mu, sd = base; out = np.zeros(len(lm), np.float32)
    z = (lm - mu) / sd
    q = np.clip(np.round(z / se + ze), -128, 127).astype(np.int8)
    for i in range(len(q)):
        it.set_tensor(ent["index"], q[i:i + 1, ..., None]); it.invoke()
        out[i] = (it.get_tensor(sai["index"])[0, 1].astype(np.float32) - zs) * ss
    return out
P = {"primeira_parte": [], "cal_5min": []}; J = []
for js, lm in janelas_da_sessao(wav, J_BASE, None, PASSO_AVAL):
    J.append(js)
    for k in P: P[k].append(prob_cnn(lm, BASES[k]))
J = np.concatenate(J); P = {k: np.concatenate(v) for k, v in P.items()}

def decisoes(p):
    """Como no firmware: a cada 10 janelas, média das 10 últimas p(anômalo) >= limiar -> nível 1."""
    n = len(p) // N_MEDIA
    return p[: n * N_MEDIA].reshape(n, N_MEDIA).mean(1) >= LIMIAR
RES = {}
for k, p in P.items():
    d = decisoes(p)
    RES[k] = {"janelas": int(len(p)), "janelas_p_maior_limiar_pct": float(100 * (p >= LIMIAR).mean()),
              "decisoes_10s": int(len(d)), "falsos_alarmes_10s": int(d.sum()), "falsos_alarmes_pct": float(100 * d.mean())}
    print(f"{k:15s} janelas p>=0,5: {RES[k]['janelas_p_maior_limiar_pct']:.2f}%   falsos alarmes (decisões de 10 s): "
          f"{RES[k]['falsos_alarmes_10s']} de {RES[k]['decisoes_10s']} ({RES[k]['falsos_alarmes_pct']:.2f}%)")

In [ ]:
# 8. LSTM nas séries ambientais (média de 10 s, janelas de 30 passos deslizantes como no firmware)
SEM_L = json.load(open(f"{LSTM}/lstm_int8_quantizacao.json"))["semente"]
lstm = tf.keras.models.load_model(f"{LSTM}/lstm_semente{SEM_L}.keras")
FAIXAS = {"temp": (10.0, 45.0), "umid": (0.0, 100.0), "dpress": (-5.0, 5.0)}
def atributos(v):
    v = np.asarray(v, np.float32); dp = v[..., 2] - v[..., 2].mean(axis=-1, keepdims=True)
    cols = [v[..., 0], v[..., 1], dp]
    return np.stack([np.clip((c - lo) / (hi - lo), 0, 1) for c, (lo, hi) in zip(cols, FAIXAS.values())], -1).astype(np.float32)
def classe_termica(t): return np.select([t > 35, t > 32, t > 27], [3, 2, 1], 0)
s10 = serie_10s(ler_amb(PASTA))
v = s10[["temp", "umid", "press"]].to_numpy(np.float32)
ini_aval = int(len(v) * FRACAO_BASE)
idx = [i for i in range(max(0, ini_aval - 29), len(v) - 29) if not np.isnan(v[i:i + 30]).any()]
Xl = np.stack([v[i:i + 30] for i in idx]); yr = classe_termica(Xl[:, -1, 0])
pl = lstm.predict(atributos(Xl), batch_size=1024, verbose=0).argmax(1)
RES["lstm"] = {"janelas": int(len(pl)), "regra_normal": int((yr == 0).sum()),
               "lstm_acima_de_normal": int((pl > 0).sum()), "lstm_acima_de_normal_pct": float(100 * (pl > 0).mean()),
               "lstm_acima_quando_regra_normal": int(((pl > 0) & (yr == 0)).sum())}
print(json.dumps(RES["lstm"], indent=1))

### ➜ DISSERTAÇÃO — Figura da seção 9.2

Painel (a): linha de base acústica do nó × linhas de base do MIMII. Painel (b): média de 10 s de p(anômalo) da CNN no trecho de avaliação, com a linha de base de 5 min e a longa.

Título sugerido: *Linha de base acústica e comportamento da CNN em operação normal*. Nota: indicar que a sessão é sintética (seção 9.1).

Arquivo: `Figura_9_2_linha_base.png / .svg` na pasta `dissertacao_cap9_sim/` (ou `dissertacao_cap9/` com dados do nó). No Word: rótulo com `SEQ Figura`, título em itálico e a nota indicada abaixo.

In [ ]:
# 9. Figuras: (a) linha de base do nó × linhas de base das máquinas do MIMII; (b) p(anômalo) na avaliação
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), constrained_layout=True)
mu, sd = BASES["sessao_inteira"]
for m, vv in qjs.get("linhas_de_base_mimii", {}).items():
    ax[0].plot(vv["media"], lw=0.8, alpha=0.6, label=f"MIMII {m}")
ax[0].fill_between(range(N_MEL), mu - sd, mu + sd, color="k", alpha=0.15, lw=0)
ax[0].plot(mu, "k", lw=1.6, label=f"Nó sensor ({SESSAO})")
ax[0].set_xlabel("Filtro mel"); ax[0].set_ylabel("Log-energia média"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) Linha de base acústica"); ax[0].yaxis.set_major_formatter(virgula)
h = J / 3600
for k, cor in [("cal_5min", "C1"), ("primeira_parte", "C0")]:
    n = len(P[k]) // N_MEDIA
    ax[1].plot(h[: n * N_MEDIA: N_MEDIA], P[k][: n * N_MEDIA].reshape(n, N_MEDIA).mean(1), lw=0.5, color=cor,
               label={"cal_5min": "Linha de base de 5 min (CAL)", "primeira_parte": f"Linha de base de {FRACAO_BASE:.0%} da sessão"}[k])
ax[1].axhline(LIMIAR, color="0.4", ls="--", lw=0.8)
ax[1].set_xlabel("Horas desde o início da gravação"); ax[1].set_ylabel("Média de 10 s de p(anômalo)")
ax[1].set_title("(b) CNN em operação normal (trecho de avaliação)"); ax[1].legend(fontsize=7)
ax[1].xaxis.set_major_formatter(virgula); ax[1].yaxis.set_major_formatter(virgula)
for ext in ["png", "svg"]:
    plt.savefig(f"{SAIDA}/Figura_linha_base_{SESSAO}.{ext}"); plt.savefig(f"{DISS}/Figura_9_2_linha_base.{ext}")
plt.show()

In [ ]:
# 10. Registro
saida = {"sessao": SESSAO, "versao_firmware": info.get("versao"), "janelas_total": N_JAN, "fracao_base": FRACAO_BASE,
         "base_csv": f"{SAIDA}/base.csv",
         "linhas_de_base": {k: {"media": v[0].tolist(), "desvio": v[1].tolist()} for k, v in BASES.items()},
         "falsos_alarmes": RES, "limiar": LIMIAR, "n_media": N_MEDIA, "passo_avaliacao": PASSO_AVAL}
json.dump(saida, open(f"{SAIDA}/linha_base_{SESSAO}.json", "w"), indent=1, ensure_ascii=False)
print("Salvo em", SAIDA)

### ➜ DISSERTAÇÃO — Tabela da seção 9.2

Falsos alarmes da CNN (linha de base de 5 min × longa) e do LSTM em operação normal. Título e nota já saem no arquivo.

Arquivo: `Tabela_9_2_falsos_alarmes.docx` na pasta `dissertacao_cap9_sim/` (ou `dissertacao_cap9/` com dados do nó). No Word: rótulo com `SEQ Tabela`, título em itálico e a nota indicada abaixo.

In [ ]:
# 11. Tabela para a dissertação: falsos alarmes em operação normal (células 7 e 8)
#     Gera CSV e .docx; no Word, cole a tabela e aplique o estilo Tabela APA e o Texto de Tabela APA.
if importlib.util.find_spec("docx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"], check=True)
from docx import Document
from docx.shared import Pt
br = lambda v, d=1: f"{v:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
h_base = J_BASE / 3600; h_aval = (N_JAN - J_BASE) / 3600
linhas = [
    ["CNN 2D (int8)", f"{JANELAS_CAL // 60} min (comando CAL)", "Média de 10 janelas de 1 s",
     br(RES["cal_5min"]["decisoes_10s"], 0), br(RES["cal_5min"]["falsos_alarmes_10s"], 0), br(RES["cal_5min"]["falsos_alarmes_pct"], 2)],
    ["CNN 2D (int8)", f"{br(h_base, 1)} h (início da sessão)", "Média de 10 janelas de 1 s",
     br(RES["primeira_parte"]["decisoes_10s"], 0), br(RES["primeira_parte"]["falsos_alarmes_10s"], 0), br(RES["primeira_parte"]["falsos_alarmes_pct"], 2)],
    ["LSTM (float32)", "Não se aplica", "Janela de 30 passos de 10 s",
     br(RES["lstm"]["janelas"], 0), br(RES["lstm"]["lstm_acima_de_normal"], 0), br(RES["lstm"]["lstm_acima_de_normal_pct"], 2)],
]
CAB = ["Modelo", "Linha de base acústica", "Unidade de decisão", "Decisões avaliadas", "Falsos alarmes", "Taxa (%)"]
pd.DataFrame(linhas, columns=CAB).to_csv(f"{SAIDA}/Tabela_falsos_alarmes_{SESSAO}.csv", index=False, sep=";", encoding="utf-8-sig")

TITULO = "Falsos alarmes da CNN e do LSTM em operação normal no ambiente representativo"
NOTA = (("Nota. Sessão sintética gerada a partir do MIMII e do Bristol (notebook 07b); não é medição no nó. " if SIM else "Nota. ") + f"Trecho de avaliação: últimas {br(h_aval, 1)} h da sessão de linha de base de {br(N_JAN / 3600, 1)} h, "
        f"sem eventos induzidos; a linha de base acústica de {br(h_base, 1)} h foi calculada no trecho anterior, sem sobreposição. "
        f"Falso alarme na CNN: média das 10 últimas probabilidades de anomalia igual ou superior a {br(LIMIAR, 1)}, como no firmware; "
        "no LSTM: classe predita acima de Normal. A taxa mede apenas o comportamento em operação normal, não a detecção de falhas. "
        "Elaborado pelo autor (2026).")
doc = Document()
st = doc.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(12)
p = doc.add_paragraph(); r = p.add_run("Tabela N"); r.bold = True
p = doc.add_paragraph(); r = p.add_run(TITULO); r.italic = True
t = doc.add_table(rows=1, cols=len(CAB))
for c, txt in zip(t.rows[0].cells, CAB): c.text = txt
for l in linhas:
    for c, txt in zip(t.add_row().cells, l): c.text = txt
for row in t.rows:
    for c in row.cells:
        for pp in c.paragraphs:
            for rr in pp.runs: rr.font.size = Pt(10)
p = doc.add_paragraph(); r = p.add_run("Nota."); r.italic = True; p.add_run(NOTA[len("Nota."):])
doc.save(f"{SAIDA}/Tabela_falsos_alarmes_{SESSAO}.docx"); doc.save(f"{DISS}/Tabela_9_2_falsos_alarmes.docx")
print(pd.DataFrame(linhas, columns=CAB).to_string(index=False)); print(); print(NOTA)

## Próximo passo no protótipo

1. Copie `base.csv` para a **raiz** do microSD usado pelo firmware de inferência (substitui o arquivo criado pelo `CAL`).
2. Ligue o nó com o firmware de inferência e confira no Monitor Serial: `>> Linha de base acústica carregada (base.csv)`. O comando `BASE` mostra os 40 valores; eles devem bater com o arquivo (4 casas decimais).
3. **Não envie `CAL` depois disso**: ele sobrescreve o `base.csv` com uma calibração de 5 min.

## O que vai para a dissertação

* Taxa de falsos alarmes da CNN e do LSTM em operação normal (célula 7 e 8), com a ressalva de que mede só a classe Normal.
* Comparação entre a linha de base de 5 min e a de longa duração (célula 7, figura b).
* A linha de base do nó comparada às do MIMII (figura a) mostra a diferença de domínio entre o ventilador do protótipo e os do dataset.